# ProductIQ Phase 2 — Step 2.5: Data Normalization

This notebook profiles cleaned AJIO data, applies deterministic normalization, and verifies that source columns remain intact.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path

import pandas as pd

from data.ingestion import AjioDataLoader
from data.preprocessing import (
    AjioDataCleaner,
    AjioDataNormalizer,
    COLOR_IS_CODED_COLUMN,
    COLOR_NORMALIZED_COLUMN,
    PRICE_ANOMALY_COLUMN,
    profile_normalization_candidates,
)
from data.validation import AjioDataValidator


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"

raw_dataframe = AjioDataLoader().load(DATASET_PATH)
validation_report = AjioDataValidator().validate(raw_dataframe)
cleaned_result = AjioDataCleaner().clean(raw_dataframe)
cleaned_dataframe = cleaned_result.dataframe

In [ ]:
print(validation_report.summary())
profile = profile_normalization_candidates(cleaned_dataframe)
print("\nNormalization profile:")
print(f"Brand unique values: {profile.brand_unique_count}")
print(f"Brand case-variant groups: {profile.brand_case_variant_groups}")
print(f"Category values: {profile.category_values}")
print(f"Color unique values: {profile.color_unique_count}")
print(f"Opaque color rows: {profile.opaque_color_row_count}")
print(f"Opaque color unique values: {profile.opaque_color_unique_count}")

In [ ]:
print("Sample brands:")
print(profile.sample_brands)
print("\nTop color values:")
print(profile.sample_colors)
print("\nSample opaque/code-like colors:")
print(profile.sample_opaque_colors)

In [ ]:
cleaned_before = deepcopy(cleaned_dataframe)
normalization_result = AjioDataNormalizer().normalize(cleaned_dataframe)
normalized_dataframe = normalization_result.dataframe
normalization_report = normalization_result.report

In [ ]:
examples = normalized_dataframe[
    normalized_dataframe["Color"] != normalized_dataframe[COLOR_NORMALIZED_COLUMN]
][["Color", COLOR_NORMALIZED_COLUMN, COLOR_IS_CODED_COLUMN]].drop_duplicates().head(20)
examples

In [ ]:
opaque_examples = normalized_dataframe[normalized_dataframe[COLOR_IS_CODED_COLUMN]][[
    "Color",
    COLOR_NORMALIZED_COLUMN,
    COLOR_IS_CODED_COLUMN,
]].drop_duplicates().head(20)
opaque_examples

In [ ]:
print(normalization_report.summary())

In [ ]:
print(f"Rows before: {len(cleaned_before)}")
print(f"Rows after: {len(normalized_dataframe)}")
print(f"Id_Product unchanged: {normalized_dataframe['Id_Product'].equals(cleaned_before['Id_Product'])}")
print(f"Brand unchanged: {normalized_dataframe['Brand'].equals(cleaned_before['Brand'])}")
print(f"Color unchanged: {normalized_dataframe['Color'].equals(cleaned_before['Color'])}")
print(f"Price anomaly unchanged: {normalized_dataframe[PRICE_ANOMALY_COLUMN].equals(cleaned_before[PRICE_ANOMALY_COLUMN])}")

In [ ]:
pd.testing.assert_frame_equal(cleaned_dataframe, cleaned_before)
print("Cleaned input DataFrame was not mutated.")
print("Raw CSV path:", DATASET_PATH)